 ```mermaid
graph TB
    Title["Dispa-SET <br>GAMS Model"]:::A
    subgraph SG1[" "]
        direction LR
        A("Configuration<br>and<br>Solver Settings"):::C
        B("Definition<br>of the<br>Dataset<br>-<br>related options"):::C
        C("Definition<br>of<br>Sets and Parameters"):::C
        D("Data<br>import"):::C
        E("Definition<br>of<br>Variables"):::C
        F("Assignment<br>of<br>Initial Values"):::C
        G("Declaration<br>and<br>definition<br>of<br>Equations"):::C
        H("Definition<br>of<br>Model"):::C
        I("Solving<br>Loop"):::C
        J("Result<br>Export"):::C
        A --> B --> C --> D --> E --> F --> G --> H --> I --> J
    end

    Title --> SG1
    
linkStyle default stroke:#FFFFFF,stroke-width:4px;
    
classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:white,font-weight:bold,font-size:30px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:white;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:white,font-weight:bold,font-size:25px
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:white,font-weight:bold,font-size:15px

class SG1 B;

 ```mermaid
flowchart TD
    D1([Start<br>Data Import]):::A --> D2[$gdxin %inputfilename%]:::B


    %% Sets Loading
    D2 --> D3["Load Core Sets"]:::B
    D3 --> D6["Load Economic Parameters"]:::C
    
            subgraph SG1[" "]
        direction TB
    %% Basic Parameters

    D6 --> D7["Load Technical Parameters:"]:::C
    
    D7 --> D8["Load Network & Location"]:::C
    
    %% Boundary Sector Parameters
    D8 --> D10["Load Boundary Sector<br>(SECTOR X)"]:::C
    
    %% First Conditional Load
    D10 --> D11{MTS ==0?}:::C
    D11 -- Yes --> D12[Load SectorXStorageInitial]:::C
    D11 -- No --> D13[Load SectorXStorageProfile]:::C
    D12 --> D13
    
    %% Second Conditional Load
    D13 --> D14{RetrieveStatus = 1?}:::C
    D14 -- Yes --> D15[Load CommittedCalc]:::C

    %% Network Data
    D14 -- No --> D17[Load PTDF Matrix]:::C
    D15 --> D17
    
    
    %% Reserve & Frequency Parameters (Conditional)
    D17 --> D19{MTS = 0?}:::C
    D19 -- Yes --> D20["Load Frequency Parameters"]:::C

        end
    D19 -- No --> D21([End]):::C
    D20 --> D21
    
classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px
classDef D fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:10px

class SG1 B;
class SG2 B;
class SG3 D;
class CC B;

%% Style the edge labels
linkStyle 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17 color:#none,stroke:#none,stroke-width:1px;
```


 ```mermaid

flowchart TD
    E1([Start<br>Definitions of Variables]):::A
    E1 --> E3[Integer / Binary<br>Variables  ]:::C
    
        subgraph S2[ ]
        %% Binary/Integer Variables
            E3 --> E6{Unit<br>Operations}:::C
                subgraph S2a[ ]
                    E6 --> E7["Unit<br>commitment<br>status"]:::C
                    E6 --> E8["Unit<br>start-up"]:::C
                    E6 --> E9["Unit<br>shut-down"]:::C
                end
                subgraph S2b[ ]
                    E7 --> E10{LP<br>Formulation?}:::C
                    E8 --> E10
                    E9 --> E10
                    E10 -->|LPFormulation = 1| E11["Unit commitment status = POSITIVE<br>Unit commitment Upper Bound = 1"]:::C
                    E10 -->|LPFormulation ≠ 1| E12["All Unit Operations Vatiables = INTEGER<br>All Unit Operations Bounds = Units Number"]:::C 
                end
            end

        subgraph S3[ ]
        %% Positive Variables
            E11 --> E4[Positive<br>Variables]:::C
            E12 --> E4  
            E4 --> E13{Power<br>&<br>Flow}:::C
                
                subgraph S3a[ ]
                    E13 --> E14["Power<br>output"]:::C
                    E13 --> E15["Boundary<br>flow"]:::C
                    E13 --> E16["Line<br>flow"]:::C
                end
    
                subgraph S3b[ ]
                    E14 --> E17{Storage<br>Systems}:::C
                    E15 --> E17
                    E16 --> E17
                    E17 --> E18["Storage<br>Charging<br>input"]:::C
                    E17 --> E19["Storage<br>level"]:::C
                    E17 --> E20["Boundary<br>sector<br>storage<br>Level"]:::C
                    E17 --> E21["Reservoir<br>spillage"]:::C
                end

                subgraph S3c[ ]
                            E18 --> E22{Reserve<br>Services}:::C
                            E19 --> E22
                            E20 --> E22
                            E21 --> E22
                    E22 --> E23["Spinning<br>reserve<br>up"]:::C
                    E22 --> E24["Spinning<br>reserve<br>down"]:::C
                    E22 --> E25["Non-spinning<br>reserve"]:::C
                end

                subgraph S3d[ ]
                            E23 --> E26{Cost Variables}:::C
                            E24 --> E26
                            E25 --> E26
                    E26 --> E27["Start-up<br>cost"]:::C
                    E26 --> E28["Ramping<br>Up/Down<br>costs"]:::C
                    E26 --> E29["Hourly<br>system<br>cost"]:::C
                end

                subgraph S3e[ ]
                            E27 --> E30{Load<br>Loss<br>Variables}:::C
                            E28 --> E30
                            E29 --> E30
                    E30 --> E31["Load<br>shedding"]:::C
                    E30 --> E32["Power<br>curtailment"]:::C
                    E30 --> E33["Max power deficit<br>/<br>Max Lost Load in Energy"]:::C
                    E30 --> E34["Reserve deficits<br>/<br>Lost Spinning Reserve Up<br>Lost Spinning Reserve Down<br>Lost Non-Spinning Reserve Up"]:::C
                end

                subgraph S3f[ ]
                            E31 --> E35{Flexible<br>Demand}:::C
                            E32 --> E35
                            E33 --> E35
                            E34 --> E35
                    E35 --> E36["Boundary Sector<br>Flexible demand"]:::C
                    E35 --> E37["Boundary Sector<br>Flexible supply"]:::C
                    E35 --> E38["Accumulated<br>oversupply<br>/<br>Deferred Demand"]:::C
                end
        
                subgraph S3g[ ]
                            E36 --> E40{MTS = 0?}:::C
                            E37 --> E40
                            E38 --> E40
                    E40 -->|Yes| E41a[Frequency Stability]:::C 
                    E41a --- E41["System Inertia<br>Primary Reserve Available<br>System Gain<br>Fast Frequency Reserve Available<br>Fast Frequency Reserve Gain"]:::C
                    E40 -->|No| E42a[Boundary Sector]:::C
                    E42a--- E42["Boundary Sector Initial state of charge<br>Boundary Sector Minimum allowed state of charge"]:::C
                end
        end

        subgraph S4[ ]
    %% Free Variables
            E41 --> E5
            E42 --> E5
            E5[Free<br>Variables]:::C --> E43{System Level}:::C
            
                subgraph S4a[ ]
            E43 --> E44[Total<br>System<br>Cost]:::C
            E43 --> E45["Objective<br>function"]:::C
            E43 --> E46["Optimality<br>gap"]:::C
                end
                
                subgraph S4b[ ]
            E44 --> E47{Power<br>Balance}:::C
            E45 --> E47
            E46 --> E47
            E47 --> E48["Demand<br>flexibility"]:::C
            E47 --> E49["Boundary<br>sector<br>power"]:::C
            E47 --> E50["Residual<br>load"]:::C
            E47 --> E51["Injected<br>power"]:::C
                end
        end
    E48 --> E52
    E49 --> E52
    E50 --> E52
    E51 --> E52([End]):::C

classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px
classDef D fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:10px

class S2 B;
class S3 B;
class S4 B;
class S2a D;
class S2b D;
class S3a D;
class S3b D;
class S3c D;
class S3d D;
class S3e D;
class S3f D;
class S3g D;
class S4a D;
class S4b D;

%% Style the edge labels
linkStyle 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73 color:#none,stroke:#none,stroke-width:1px;
```

 ```mermaid
flowchart TD

        E1(["Start<br>Assignment of Initial Values"]):::A --> E2{{"Set Initial<br>Commitment Status"}}:::C
    
    subgraph S1[" "]
        E2 --> E3["All Units: Set as OFF by Default"]:::C
        E2 --> E4["Units with Initial Power > 0: Set as ON"]:::C
    end
    
    subgraph S2[" "]
        E3 --> E5{{"Calculate<br>Power Parameters"}}:::C
        E4 --> E5
        E5 --> E6["Calculate Minimum<br>Stable Power"]:::C
        E5 --> E7["Calculate Maximum<br>Load Availability"]:::C
    end
    
    subgraph S3[" "]
        E6 --> E8{{"Check Quick<br>Start Capability"}}:::C
        E7 --> E8
        E8 --> E9{"Can Ramp Up<br>Fast Enough?"}:::C
        E9 -->|Yes| E10["Quick Start Power<br>=<br>Full Capacity"]:::C
        E9 -->|No| E11["Quick Start Power<br>=<br>Zero"]:::C
    end
    
    subgraph S4[" "]
        E10 --> E12{{"Calculate<br>Ramp Limits"}}:::C
        E11 --> E12
        E12 --> E13["Set Start-Up<br>Ramp Limit"]:::C
        E12 --> E14["Set Shut-Down<br>Ramp Limit"]:::C
    end
    
    subgraph S5[" "]
        E13 --> E15{{"Determine<br>Must-Run Status"}}:::C
        E14 --> E15
        E15 --> E16["Standard Must-Run<br>=<br>Minimum Stable Power"]:::C
        E15 --> E17["Special Must-Run<br>for<br>Certain Technologies"]:::C
    end
    
    subgraph S6[" "]
        E16 --> E18{{"Set<br>Reserve Parameters"}}:::C
        E17 --> E18
        E18 --> E19["Reserve Share from<br>Offline Quick Start Units"]:::C
    end
    
    subgraph S7[" "]
        E19 --> E20{{"Configure<br>Flexible Demand"}}:::C
        E20 --> E21["Find Maximum<br>Flexible Demand"]:::C
        E20 --> E22["Set Maximum<br>Oversupply Allowed"]:::C
        E20 --> E23["Initialize Accumulated<br>Oversupply to Zero"]:::C
    end
    
    subgraph S8[" "]
        E21 --> E24{{"Set Time<br>Configuration"}}:::C
        E22 --> E24
        E23 --> E24
        E24 --> E25["Define Simulation<br>Time Step"]:::C
    end
    

        E25 --> E26(["End"]):::C

classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px
classDef D fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:10px

class S1 B;
class S2 B;
class S3 B;
class S4 B;
class S5 B;
class S6 B;
class S7 B;
class S8 B;
class S9 B;


%% Style the edge labels
linkStyle 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,30 color:#none,stroke:#none,stroke-width:1px;


 ```mermaid
flowchart TD
    A0([Start<br>Equation Definitions]):::A --> A2

    subgraph S1[" "]
        A2{"LP Formulation = 1?"}:::C
        A3["EQ_SystemCost<br>Linear programming cost calculation<br>(relaxed commitment)"]:::C
        A4["EQ_SystemCost<br>Integer programming cost calculation<br>(full unit commitment)"]:::C
        A2 -->|Yes| A3
        A2 -->|No| A4
        A3 & A4 --- A1["EQ_Objective_function<br>Minimizes total system cost"]:::C
    end

    subgraph S2[" "]
        A1 --> A5{{"Commitment Constraints"}}:::C
        A5 --> A6["EQ_MinUpTime<br>Enforces minimum uptime duration<br>after unit startup"]:::C
        A5 --> A7["EQ_MinDownTime<br>Enforces minimum downtime duration<br>after unit shutdown"]:::C
        A5 --> A8["EQ_RampUp_TC<br>Limits maximum power increase<br>between time steps"]:::C
        A5 --> A9["EQ_RampDown_TC<br>Limits maximum power decrease<br>between time steps"]:::C
    end

    subgraph S3[" "]
        A6 & A7 & A8 & A9 --> A10a{{"Cost Equations"}}:::C
        A10a --- A10["EQ_CostStartUp<br>Calculates startup costs<br>per unit per hour"]:::C
        A10a --> A11["EQ_CostShutDown<br>Calculates shutdown costs<br>per unit per hour"]:::C
        A10a --> A12["EQ_CostRampUp<br>Calculates ramping up costs<br>for power increases"]:::C
        A10a --> A13["EQ_CostRampDown<br>Calculates ramping down costs<br>for power decreases"]:::C
    end

    subgraph S4[" "]
        A10 & A11 & A12 & A13 --> A14{"Retrieve Initial<br>Commitment Status = 1?"}:::C
        A14 -->|Yes| A15["EQ_CommittedCalc<br>Retrieves commitment status<br>from previous solution"]:::C
        A14 -->|No| A16[SkipRetrieve]:::C
    end

    subgraph S5[" "]
        A15 & A16 --> A17a{{"Power Balance"}}:::C
        A17a --- A17["EQ_Residual_Load<br>Calculates net load after<br>accounting for injections"]:::C
        A17a --> A18["EQ_Demand_balance_DA<br>Ensures generation matches<br>day-ahead demand"]:::C
    end

    subgraph S6[" "]
        A17 & A18 --> A19{"Flexible Demand Active<br>=1?"}:::C
        A19 -->|Yes| A20["EQ_Flexible_Demand<br>Models demand flexibility<br>as virtual storage"]:::C
        
            subgraph S6a[" "]
                A20 --> A21["EQ_Flexible_Demand_max<br>Limits total flexible demand<br>capacity"]:::C
                A21 --> A22["EQ_Flexible_Demand_Modulation_Min<br>Minimum downward demand adjustment"]:::C
                A22 --> A23["EQ_Flexible_Demand_Modulation_max<br>Maximum upward demand adjustment"]:::C
            end
        A19 -->|No| A24["EQ_No_Flexible_Demand<br>Disables flexible demand<br>when inactive"]:::C
            
    end

    subgraph S7[" "]
        A23 & A24 --> A25a{{"Reserve Markets"}}:::C
        A25a --- A25["EQ_Demand_balance_2U<br>Spinning reserve<br>up requirement"]:::C
        A25a --> A26["EQ_Tot_Demand_2U<br>System-wide spinning<br>reserve sum"]:::C
        A25a --> A27["EQ_Demand_balance_3U<br>Non-spinning<br>reserve requirement"]:::C
        A25a --> A28["EQ_Demand_balance_2D<br>Spinning reserve<br>down requirement"]:::C
        A25a --> A29["EQ_Curtailed_Power<br>Accounts for renewable<br>energy curtailment"]:::C
        A25a --> A30["EQ_Reserve_2U_capability<br>Unit spinning<br>reserve up capacity"]:::C
        A25a --> A31["EQ_Reserve_2D_capability<br>Unit spinning<br>reserve down capacity"]:::C
        A25a --> A32["EQ_Reserve_3U_capability<br>Unit non-spinning<br>reserve capacity"]:::C
    end

    subgraph S8[" "]
        A25 & A26 & A27 & A28 & A29 & A30 & A31 & A32 --> A33{"MTS = 0?"}:::C
        A33 -->|Yes| A34a
        A33 -->|Yes| A36a
        A33 -->|Yes| A38a
        A33 -->|Yes| A44a
        A33 -->|No| A35[Skip MTS]:::C
        
                A34a{{"Power Tracking"}}:::C
                A34["EQ_PowerLoss<br>Tracks maximum power loss<br>for stability analysis"]:::C

                A36a{{"Inertia Constraints"}}:::C
                A36["EQ_SysInertia<br>Calculates system inertia<br>from committed units"]:::C
                A37["EQ_Inertia_limit<br>Enforces minimum system<br>inertia requirement"]:::C

                A38a{{Primary Reserve}}:::C
                A38["EQ_SystemGain<br>Calculates frequency response<br>capability"]:::C
                A39["EQ_SystemGain_limit<br>Limits frequency response<br>capability"]:::C
                A40["EQ_PrimaryReserve_Available<br>Primary reserve provision"]:::C
                A41["EQ_PrimaryReserve_Capability<br>Unit primary reserve capacity"]:::C
                A42["EQ_PrimaryReserve_Boundary<br>Primary reserve limits"]:::C
                A43["EQ_Demand_balance_PrimaryReserve<br>Primary reserve requirement"]:::C

                A44a{{"Fast Frequency Reserve"}}:::C
                A44["EQ_FFRGain<br>Fast frequency response<br>capability"]:::C
                A45["EQ_FFRGain_limit<br>FFR capability limits"]:::C
                A46["EQ_FFR_Available<br>Fast frequency reserve provision"]:::C
                A47["EQ_FFR_Capability<br>Unit FFR capacity"]:::C
                A48["EQ_FFR_Boundary<br>FFR provision limits"]:::C
                A49["EQ_Demand_balance_FFR<br>FFR requirement"]:::C
                
            subgraph S8a[" "]
                A34a --> A34
            end
            subgraph S8b[" "]
                A36a --> A36
                A36 --> A37
                
            end
            subgraph S8c[" "]
                A38a --> A38
                A38 --> A39
                A39 --> A40
                A40 --> A41
                A41 --> A42
                A42 --> A43
                
            end
            subgraph S8d[" "]
                A44a --> A44
                A44 --> A45
                A45 --> A46
                A46 --> A47
                A47 --> A48
                A48 --> A49
            end
    end

    subgraph S9[" "]
        A35 & A34 & A37 & A43 & A49 --> A50a{{"Core Constraints"}}:::C
        A50a --> A50["EQ_Power_must_run<br>Minimum stable<br>generation level"]:::C
        A50a --> A51["EQ_Power_available<br>Maximum<br>available capacity"]:::C
        A50a --> A52["EQ_Storage_minimum<br>Minimum<br>storage level"]:::C
        A50a --> A53["EQ_Storage_alert<br>Storage alert level<br>with violation"]:::C
        A50a --> A54["EQ_Storage_flood_control<br>Maximum<br>storage level"]:::C
        A50a --> A55["EQ_Storage_level<br>Storage<br>capacity limit"]:::C
        A50a --> A56["EQ_Storage_input<br>Storage<br>charging limit"]:::C
        A50a --> A57["EQ_Storage_MaxDischarge<br>Maximum<br>discharge rate"]:::C
        A50a --> A58["EQ_Storage_MaxCharge<br>Maximum<br>charging rate"]:::C
        A50a --> A59["EQ_Storage_balance<br>Storage<br>energy conservation"]:::C
        A50a --> A60["EQ_Storage_boundaries<br>Final<br>storage level"]:::C
        A50a --> A61["EQ_Storage_Cyclic<br>Cyclic<br>storage conditions"]:::C
    end

    subgraph S10[" "]
        A50 & A51 & A52 & A53 & A54 & A55 & A56 & A57 & A58 & A59 & A60 & A61 --> A62a{{"Network Equations"}}:::C
        A62a --- A62["EQ_Emission_limits<br>Pollution emission caps"]:::C
        A62a --> A63["EQ_DC_Power_Flow<br>Linearized power flow model"]:::C
        A62a --> A64["EQ_Total_Injected_Power<br>Node injection balance"]:::C
        A62a --> A65["EQ_Flow_limits_lower<br>Minimum line flow"]:::C
        A62a --> A66["EQ_Flow_limits_upper<br>Maximum line flow"]:::C
        A62a --> A67["EQ_BS_Flow_limits_lower<br>Minimum boundary flow"]:::C
        A62a --> A68["EQ_BS_Flow_limits_upper<br>Maximum boundary flow"]:::C
        A62a --> A69["EQ_BS_Spillage_limits_upper<br>Boundary sector spillage limit"]:::C
    end

    subgraph S11[" "]
        A62 & A63 & A64 & A65 & A66 & A67 & A68 & A69 --> A69a{{"Operational Constraints"}}:::C
        A69a --> A70["EQ_Force_Commitment<br>Mandatory unit commitment"]:::C
        A69a --> A71["EQ_Force_DeCommitment<br>Mandatory unit decommitment"]:::C
        A69a --> A72["EQ_LoadShedding<br>Emergency load reduction limits"]:::C
    end

    subgraph S12[" "]
        A70 & A71 & A72 --> A73a{{"CHP Equations"}}:::C
        A73a --> A73["EQ_CHP_extraction<br>Extraction CHP power-heat coupling"]:::C
        A73a --> A74["EQ_CHP_extraction_Pmax<br>Extraction CHP maximum power"]:::C
        A73a --> A75["EQ_CHP_backpressure<br>Backpressure CHP operation"]:::C
        A73a --> A76["EQ_CHP_max_heat<br>CHP maximum heat output"]:::C
    end

    subgraph S13[" "]
        A73 & A74 & A75 & A76 --> A77a{{"Boundary Sector"}}:::C
        A77a --> A77["EQ_Power_Balance_of_P2X_units<br>Power-to-X conversion balance"]:::C
        A77a --> A78["EQ_Power_Balance_of_X2P_units<br>X-to-Power conversion balance"]:::C
        A77a --> A79["EQ_Max_Power_Consumption_of_BS_units<br>Boundary sector power limits"]:::C
        A77a --> A80["EQ_BS_Demand_balance<br>Boundary sector energy balance"]:::C
    end

    subgraph S14[" "]
        A77 & A78 & A79 & A80 --> A81a{{"Boundary Sector<br>Flex and Conversion"}}:::C
        A81a --> A81["EQ_Tot_Flex_Demand_BS<br>Total boundary<br>flex demand"]:::C
        A81a --> A82["EQ_Max_Flex_Capacity_BS<br>Boundary flex<br>demand limits"]:::C
        A81a --> A83["EQ_BS_Flex_Demand<br>Boundary sector<br>flexible demand"]:::C
        A81a --> A84["EQ_Tot_Flex_Supply_BS<br>Total boundary<br>flex supply"]:::C
        A81a --> A85["EQ_Max_Flex_Supply_BS<br>Boundary flex<br>supply limits"]:::C
        A81a --> A86["EQ_P2X_Power_Balance<br>Power-to-X<br>balance equation"]:::C
        A81a --> A87["EQ_X2P_Power_Consumption<br>X-to-Power<br>consumption"]:::C
        A81a --> A88["EQ_Max_Power_Consumption<br>Maximum<br>conversion capacity"]:::C
    end
    A81 --- A89([End]):::C
    A82 --- A89
    A83 --- A89
    A84 --- A89
    A85 --- A89
    A86 --- A89
    A87 --- A89
    A88 --- A89

classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px
classDef D fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:10px

class S1 B;
class S2 B;
class S3 B;
class S4 B;
class S5 B;
class S6 B;
class S6a D;
class S7 B;
class S8 B;
class S8a D;
class S8b D;
class S8c D;
class S8d D;
class S9 B;
class S10 B;
class S11 B;
class S12 B;
class S13 B;
class S14 B;

%% Style the edge labels
linkStyle 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45 color:#none,stroke:#none,stroke-width:1px;

 ```mermaid
flowchart TD
    F1([Start<br>Definition of the Model]):::A --> F3["Hourly System Cost<br><br>EQ_SystemCost"]:::C
    subgraph S0["Objective & Cost"]
        F3 --> F4{"LP<br>Formulation =1?"}:::C
        F4 -->|No| F5["Startup & Shutdown Costs<br><br>EQ_CostStartUp<br>EQ_CostShutDown"]:::C
        F5 --> F6["Total Optimization Cost<br><br>EQ_Objective_function"]:::C
        F4 -->|Yes| F6
        F3 --> F7["Ramping Costs<br><br>EQ_CostRampUp<br>EQ_CostRampDown"]:::C
        F7 --> F6
    end

    subgraph S1["Commitment Constraints"]
        F6 --> F8["Unit Commitment<br>Ramp Rate Limits<br><br>EQ_Commitment<br>EQ_RampUp_TC<br>EQ_RampDown_TC"]:::C
        F8 --> F9{"LP<br>Formulation =1?"}:::C
        F9 -->|No| F10["Minimum Up & Down Time<br><br>EQ_MinUpTime<br>EQ_MinDownTime"]:::C
        F10 --> F11
        F9 -->|Yes| F11
    end

    subgraph S2["Power & Demand Balance"]
        F11["Residual Load<br><br>EQ_Residual_Load"]:::C --> F12["Day-Ahead Demand Balance<br><br>EQ_Demand_balance_DA"]:::C
        F12 --> F13{"LP<br>Formulation =1?"}:::C
        F13 -->|No| F14["Must-Run Power<br><br>EQ_Power_must_run"]:::C
        F14 --> F15
        F13 -->|Yes| F15
    end

    subgraph S3["Flexible Demand & Boundary Sector"]
        F15{Flexible Demand Active =1?}:::C --> |Yes| F16["Flexible Demand<br>Max Flexibility<br><br>EQ_Flexible_Demand<br>EQ_Flexible_Demand_Max"]:::C
        F15 -->|No| F17["No Flexible Demand<br><br>EQ_No_Flexible_Demand"]:::C
        F16 & F17 --> F18["Boundary Sector Flexibility<br>Max Power Consumption<br><br>EQ_Tot_Flex_Demand<br>EQ_Max_Flex_Demand<br>EQ_Max_Flex_Supply<br>EQ_Max_Power_Consumption_of_BS_units"]:::C
    end

    subgraph S4["Reserves & Curtailment"]
        F18 --> F19["Spinning & Non-Spinning Reserve<br>Total Reserve Demand<br><br>EQ_Demand_balance_2U<br>EQ_Demand_balance_2D<br>EQ_Demand_balance_3U<br>EQ_Tot_Demand_2U"]:::C
        F19 --> F20["Reserve Capability<br>VRES Curtailment<br><br>EQ_Reserve_2U_capability<br>EQ_Reserve_2D_capability<br>EQ_Reserve_3U_capability<br>EQ_Curtailed_Power"]:::C
    end

    subgraph S5["CHP & P2X Equations"]
        F20 --> F21["CHP Operation Limits<br>Heat-to-Power Constraints<br><br>EQ_CHP_extraction_Pmax<br>EQ_CHP_extraction<br>EQ_CHP_backpressure<br>EQ_CHP_max_heat<br>EQ_2U_limit_chp<br>EQ_2D_limit_chp<br>EQ_3U_limit_chp"]:::C
        F21 --> F22["P2X Power Balance<br>X2P Consumption<br>Max P2X Load<br><br>EQ_P2X_Power_Balance<br>EQ_X2P_Power_Consumption<br>EQ_Max_Power_Consumption"]:::C
    end

    subgraph S6["Storage & Boundary Storage"]
        F22 --> F23["Storage Limits & Levels<br>Charge/Discharge Balance<br><br>EQ_Storage_minimum<br>EQ_Storage_alert<br>EQ_Storage_flood_control<br>EQ_Storage_level<br>EQ_Storage_input<br>EQ_Storage_balance<br>EQ_Storage_boundaries<br>EQ_Storage_MaxCharge<br>EQ_Storage_MaxDischarge"]:::C
        F23 --> F24{"Rolling Horizon?<br>MTS =1?"}:::C
        F24 -->|Yes| F25["Cyclic Storage (MTS)<br><br>EQ_Storage_Cyclic"]:::C
        F25 --> F26
        F24 -->|No| F26
        F26["Boundary Sector Storage<br>Level, Charge & Discharge<br><br>EQ_Boundary_Sector_Storage_MaxDischarge<br>EQ_Boundary_Sector_Storage_MaxCharge<br>EQ_Boundary_Sector_Storage_PowerMax<br>EQ_Boundary_Sector_Storage_PowerMin<br>EQ_Boundary_Sector_Storage_minimum<br>EQ_Boundary_Sector_Storage_level<br>EQ_Boundary_Sector_Storage_alert<br>EQ_Boundary_Sector_Flood_Control<br>EQ_Boundary_Sector_Storage_balance<br>EQ_Boundary_Sector_Storage_boundaries"]:::C --> F27{"Rolling Horizon?<br>MTS =1?"}:::C
        F27 -->|Yes| F28["Cyclic Boundary Storage<br><br>EQ_Boundary_Sector_Storage_Cyclic"]:::C
        F28 --> F29
        F27 -->|No| F29
    end

    subgraph S7["Network & Operational"]
        F29["Power Availability<br>Load Shedding<br>Flow Limits<br>DC Power Flow<br><br>EQ_Power_available<br>EQ_LoadShedding<br>EQ_Flow_limits_lower<br>EQ_Flow_limits_upper<br>EQ_BS_Flow_limits_lower<br>EQ_BS_Flow_limits_upper<br>EQ_Total_Injected_Power<br>EQ_DC_Power_Flow"]:::C --> F30{"Use Prior Commitment?<br>RetrieveStatus =1?"}:::C
        F30 -->|Yes| F31["Fixed Initial Commitment<br><br>EQ_CommittedCalc"]:::C
        F31 --> F32
        F30 -->|No| F32
    end

    subgraph S8["System Services (MTS = 0)"]
        F32{"Frequency-Constrained Mode?<br>MTS =0?"}:::C -->|Yes| F33["Inertia & Frequency Response<br>Primary & FFR Reserves<br>Power Loss<br><br>EQ_SysInertia<br>EQ_Inertia_limit<br>EQ_SystemGain<br>EQ_SystemGain_limit<br>EQ_PrimaryReserve_Available<br>EQ_PrimaryReserve_Capability<br>EQ_PrimaryReserve_Boundary<br>EQ_Demand_balance_PrimaryReserve<br>EQ_FFRGain<br>EQ_FFRGain_limit<br>EQ_FFR_Available<br>EQ_FFR_Capability<br>EQ_FFR_Boundary<br>EQ_Demand_balance_FFR<br>EQ_PowerLoss"]:::C
        F33 --> F34
        F32 -->|No| F34
    end

    F34([Model Configuration]):::C --> F35([End]):::C

classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px
classDef D fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:10px

class S0 B;
class S1 B;
class S2 B;
class S3 B;
class S4 B;
class S5 B;
class S6 B;
class S7 B;
class S8 B;
class S9 B;
class S10 B;
class S11 B;
class S12 B;
class S13 B;
class S14 B;

%% Style the edge labels
linkStyle 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,30,31,32,33,34,35,36,37,38,39,40,41 color:#none,stroke:#none,stroke-width:1px;

 ```mermaid
flowchart TD
    H1(["Start<br>Solving Loop"]):::A --> H2["Calculate Total Days<br><br>Calculate ndays<br/>ndays = floor(card(h)*TimeStep/24)"]:::C

    subgraph SG1["Validation Phase"]
        H2 --> H3{"Look Ahead<br>Too Long?<br><br>LookAhead<br>><br>ndays-1?"}:::C
        H3 -->|Yes| H4["ABORT: Look ahead period<br>longer than simulation length"]:::C
        H3 -->|No| H5{"Look Ahead Not Compatible with Time Step?<br><br>LookAhead*24 mod TimeStep ≠ 0?"}:::C
        H5 -->|Yes| H6["Abort Execution:<br/>Time Step Mismatch<br><br>ABORT: Look ahead period not multiple of TimeStep"]:::C
        H5 -->|No| H7{"Horizon Length Not Compatible with Time Step?<br><br>Length*24 mod TimeStep ≠ 0?"}:::C
        H7 -->|Yes| H8["Abort Execution:<br/>Horizon Length Mismatch<br><br>ABORT: Rolling horizon length not multiple of TimeStep"]:::C
        H7 -->|No| H9
        H9["Initialize Loop Parameters and Display Configuration<br><br>• Set tmp = {model, solver}<br/>• Parameter status(tmp,h)<br/>• Scalar starttime<br/>• Set days, display parameters"]:::C
    end

    subgraph SG3["Rolling Horizon Optimization Loop"]
        H10["Start Rolling Horizon Loop Through All Days<br><br>FOR day = 1 TO ndays-LookAhead BY RollingHorizon Length"]:::C
        
        subgraph SG3A["Time Window Setup"]
            H11["Calculate Time Windows for Current Optimization<br><br>• FirstHour = (day-1)*24/TimeStep+1<br/>• LastHour = min(card(h), FirstHour + (Length+LookAhead)*24/TimeStep - 1)<br/>• LastKeptHour = LastHour - LookAhead*24/TimeStep"]:::C
            H12["Set Current Optimization Horizon<br><br>• i(h) = no<br/>• i(h) = yes for FirstHour ≤ ord(h) ≤ LastHour<br/>• Display day, FirstHour, LastHour, LastKeptHour"]:::C
        end
        
        subgraph SG3B["Storage Requirements Setup"]
            H13["Define Storage Final Requirements<br><br>• StorageFinalMin(s) = sum(...)<br/>• StorageFinalMin(chp) = sum(...)"]:::C
            H14{"Multi-Time Scale Mode Active?<br><br>MTS = 0?"}:::C
            H15["Set Sector Storage Requirements<br><br>SectorXStorageFinalMin(nx) = sum(...)"]:::C
        end
        
        subgraph SG3C["Model Solving"]

            H17{"Linear Programming Formulation?<br><br>LPFormulation = 1?"}:::C --> |Yes| H18["Solve as Linear Program<br><br>SOLVE UCM_SIMPLE USING LP MINIMIZING SystemCostD"]:::C
            H17 -->|No| H19["Solve as Mixed Integer Program<br><br>SOLVE UCM_SIMPLE USING MIP MINIMIZING SystemCostD"]:::C
            H20["Record Solution Status Information<br><br>• status(model,i) = UCM_SIMPLE.Modelstat<br/>• status(solver,i) = UCM_SIMPLE.Solvestat"]:::C
        end
        
        subgraph SG3D["Results Processing & State Updates"]
            H21["Update Initial Conditions for Next Iteration<br><br>• CommittedInitial(au) = Committed.L at LastKeptHour<br/>• PowerInitial(u) = Power.L at LastKeptHour<br/>• StorageInitial(s,chp) = StorageLevel.L at LastKeptHour"]:::C
            H22{"Multi-Time Scale Mode Active?<br><br>MTS = 0?"}:::C
            H23["Update Sector Storage States<br><br>SectorXStorageInitial(nx) = SectorXStorageLevel.L at LastKeptHour"]:::C
            H24["Update Flexible Demand States<br><br>• SectorXFlexDemandInputInitial(nx)<br/>• SectorXFlexSupplyInputInitial(nx)"]:::C
            H25{"Flexible Demand Activated?<br><br>ActivateFlexibleDemand = 1?"}:::C
            H26["Update Accumulated Oversupply Information<br><br>AccumulatedOverSupply_inital(n) = AccumulatedOverSupply.L at LastKeptHour"]:::C
        end
        
        subgraph SG3E["Results Assignment & Error Calculation"]
            H27["Assign Results to Hourly Arrays<br><br>• StorageSlack.L = Waterslack.L<br/>• StorageLevelViolation_H.L = StorageLevelViolation.L<br/>• SectorXStorageLevelViolation_H.L<br/>• ObjectiveFunction.L = SystemCostD.L"]:::C
            H28["Calculate Total Error Metrics<br><br>Calculate Error.L = Σ(CostLoadShedding*ShedLoad.L + ValueOfLostLoad*(LL_MaxPower + LL_MinPower) + 0.8*ValueOfLostLoad*(LL_2U + LL_2D + LL_3U) + 0.7*ValueOfLostLoad*(LL_RampUp + LL_RampDown))"]:::C
            H29["Calculate Optimality and Error Gaps<br><br>• OptimalityGap.L = objVal - objEst<br/>• OptimizationError.L = Error.L - OptimalityGap.L"]:::C
        end
        
        subgraph SG3F["Loop Control"]
            H30{"More Days to Process?<br><br>More days?"}:::C
        end
    end

    subgraph SG4["Final Results"]
        H31["Display All Final Results<br><br>PowerX.L, Flow.L, Power.L, Committed.L, ShedLoad.L, CurtailedPower.L, StorageLevel.L, SystemCost.L, LL_MaxPower.L, etc."]:::C
        H32(["Loop Complete"]):::C
    end


    H32 --- H33(["End"]):::C
    
    %% Connections between subgraphs
    H9 --> H10
    H10 --> H11
    H11 --> H12
    H12 --> H13
    H13 --> H14
    H14 -->|Yes| H15
    H14 -->|No| H17
    H15 --> H17
    H18 --> H20
    H19 --> H20
    H20 --> H21
    H21 --> H22
    H22 -->|Yes| H23
    H22 -->|No| H24
    H23 --> H24
    H24 --> H25
    H25 -->|Yes| H26
    H25 -->|No| H27
    H26 --> H27
    H27 --> H28
    H28 --> H29
    H29 --> H30
    H30 -->|Yes| H10
    H30 -->|No| H31
    H31 --> H32
    
    %% Error paths
    H4 --> H33
    H6 --> H33
    H8 --> H33


classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px
classDef D fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:10px

class SG1 B;
class SG2 B;
class SG3 B;
class SG3A D;
class SG3B D;
class SG3C D;
class SG3D D;
class SG3E D;
class SG3F D;
class SG4 B;
class SG5 B;
class SG6 B;
class SG7 B;
class SG8 B;
class SG9 B;
class SG10 B;
class SG11 B;
class SG12 B;
class SG13 B;
class SG14 B;

%% Style the edge labels
linkStyle 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,30,31,32,33,34,35,36,37,38 color:#none,stroke:#none,stroke-width:1px;

 ```mermaid
graph TB
   %% General Structure of the Dispa-SET GAMS Model
   %% Main Components
   A([Start<br>Dispa-SET GAMS Model]):::A --> S1 --> S2 --> S3 --> S4 --> S4_5 --> S5 --> S6 --> S7 --> S8 --> S8_5 --> S9 --> I([End]):::C

   %% Model Title and Global Settings
   direction LR
   subgraph S1["Model Title and Global Settings"]
       A1{{"Model Title and Global Settings"}}:::C --> |"Define"|A1_1["Title: UCM model"]:::C
       A1 --> |"Set"|A1_2["Global Options"]:::C
   end

   %% Model Configuration Options
   direction LR
   subgraph S2["Model Configuration Options"]
       A2{{"Model Configuration Options"}}:::C --> |"Define"|A2_1["Input File Name"]:::C
       A2 --> |"Set"|A2_2["Formulation Type"]:::C
       A2 --> |"Set"|A2_3["Retrieve Status"]:::C
       A2 --> |"Set"|A2_4["Activate Flexible Demand"]:::C
       A2 --> |"Set"|A2_5["Activate Advanced Reserves"]:::C
       A2 --> |"Set"|A2_6["Frequency Constraints (MTS)"]:::C
   end

   %% Set Definitions
   direction LR
   subgraph S3["Set Definitions"]
       B{{"Set Definitions"}}:::C --> |"Define sets for"|B1["Markets"]:::C
       B --> |"Define sets for"|B2["Units"]:::C
       B --> |"Define sets for"|B3["Fuels"]:::C
       B --> |"Define sets for"|B4["Technologies"]:::C
       B --> |"Define sets for"|B5["Nodes"]:::C
       B --> |"Define sets for"|B6["Lines"]:::C
       B --> |"Define sets for"|B7["Boundary Sector (Sector X)"]:::C
   end

   %% Parameter Definitions
   direction LR
   subgraph S4["Parameter Definitions"]
       C{{"Parameter Definitions"}}:::C --> |"Define parameters for"|C1["Operational Parameters"]:::C
       C --> |"Define parameters for"|C2["Economic Parameters"]:::C
       C --> |"Define parameters for"|C3["Storage Parameters"]:::C
       C --> |"Define parameters for"|C4["Flexible Demand Parameters"]:::C
       C --> |"Define parameters for"|C5["Boundary Sector Parameters"]:::C
       C --> |"Define parameters for"|C6["Frequency Constraints (MTS)"]:::C
   end

   %% Data Import
   direction LR
   subgraph S4_5["Data Import"]
       D4_5{{"Data Import"}}:::C --> |"Load from"|D4_5_1["$gdxin %inputfilename%"]:::C
       D4_5_1 --> |"Load sets"|D4_5_2["Core model dimensions (mk, n, nx, l, etc.)"]:::D
       D4_5_1 --> |"Load parameters"|D4_5_3["Operational & economic inputs"]:::D
       D4_5_1 --> |"Load conditional"|D4_5_4["Boundary sector parameters"]:::D
       D4_5_1 --> |"Load conditional"|D4_5_5["Network data (PTDF)"]:::D
       D4_5_1 --> |"Load conditional"|D4_5_6["Reserve & frequency parameters"]:::D
   end

   %% Variable Definitions
   direction LR
   subgraph S5["Variable Definitions"]
       D{{"Variable Definitions"}}:::C --> |"Define variables for"|D1["Binary Variables"]:::C
       D --> |"Define variables for"|D2["Continuous Variables"]:::C
       D --> |"Define variables for"|D3["Positive Variables"]:::C
       D --> |"Define variables for"|D4["Boundary Sector Variables"]:::C
       D --> |"Define variables for"|D5["Frequency Constraint Variables"]:::C
   end

   %% Equation Definitions
   direction LR
   subgraph S6["Equation Definitions"]
       E{{"Equation Definitions"}}:::C --> |"Define equations for"|E1["Objective Function"]:::C
       E --> |"Define equations for"|E2["Commitment Constraints"]:::C
       E --> |"Define equations for"|E3["Ramp Constraints"]:::C
       E --> |"Define equations for"|E4["Demand Balance Constraints"]:::C
       E --> |"Define equations for"|E5["Storage Constraints"]:::C
       E --> |"Define equations for"|E6["Reserve Constraints"]:::C
       E --> |"Define equations for"|E7["Boundary Sector Constraints"]:::C
       E --> |"Define equations for"|E8["Frequency Constraint Equations (MTS)"]:::C
   end

   %% Model Definition
   direction LR
   subgraph S7["Model Definition"]
       F{{"Model Definition"}}:::C --> |"Define model using"|F1["Model Declaration"]:::C
       F --> |"Include equations in"|F2["Model Equations"]:::C
   end

   %% Solve Statement
   direction LR
   subgraph S8["Solve Statement"]
       G{{"Solve Statement"}}:::C --> |"Solve model using"|G1["Solve Using LP / MIP"]:::C
       G --> |"Set optimization options"|G2["Optimization Options"]:::C
   end

   %% Rolling Horizon Loop
   direction LR
   subgraph S8_5["Rolling Horizon Loop"]
       H{{"Rolling Horizon Loop"}}:::C --> |"For each day in horizon"|H1["Update FirstHour, LastHour, LastKeptHour"]:::C
       H --> |"Update"|H2["Update i(h) for current window"]:::C
       H --> |"Set"|H3["Set StorageFinalMin, SectorXStorageFinalMin"]:::C
       H --> |"Solve"|H4["Solve UCM_SIMPLE"]:::C
       H --> |"Update initial conditions"|H5["Update CommittedInitial, PowerInitial, StorageInitial"]:::C
       H --> |"Loop"|H6["Repeat until end of horizon"]:::C
   end

   %% Result Export
   direction LR
   subgraph S9["Result Export"]
       I{{"Result Export"}}:::C --> |"Export results to"|I1["Export Results to GDX"]:::C
       I --> |"Display results"|I2["Display Results"]:::C
       I --> |"Key outputs"|I3["OutputPower, OutputStorageLevel, OutputSystemCost, etc."]:::C
   end

classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
   
   classDef E fill:#004C99,stroke:#none,stroke-width:2px,color:none,font-weight:bold,font-size:16px;

classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;

   class S1 Z;
   class S2 Z;
   class S3 Z;
   class S4 Z;
   class S4_5 Z;
   class S5 Z;
   class S6 Z;
   class S7 Z;
   class S8 Z;
   class S8_5 Z;
   class S9 Z;

%% Default arrow style 
linkStyle default stroke:#FFFFFF,stroke-width:2.5px;

%%classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
%%classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
%%classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
%%classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;

%%classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;

%%class SA Z; 
%%class SB Z;
%%class SC Z;
%%class SD Z;

%% Default arrow style 

%%linkStyle 0,1,2 stroke:#004C99,stroke-width:2px;

 ```mermaid
graph LR
   
    subgraph SA["Model Title and Global Settings"]
    direction LR
   
    A0{{"Model Title<br>and<br>Global Settings"}}:::A

    A0 --- SB 
    SB --- SC 
    SC --- SD
   
    A0 --> |Define|B1
    A0 --> B2


   
        subgraph SB[ ]
        direction TB
               
        B1["UCM model Title"]:::B
        B2["Global Options"]:::B


       
       end

       subgraph SC[ ]
       direction TB
       
        C1["Enable 16 parallel threads for computation"]:::C
        C2["Set iteration limit to 1 billion"]:::C
        C3["Set resource limit to 10 billion units"]:::C
        C4["Use Gurobi as the optimization solver"]:::C
        C5["Minimize listing file output"]:::C
        C6["Display all equations without row limit"]:::C
        C7["Display all variables without column limit"]:::C
        C8["Disable solver solution output printing"]:::C
        C9["Disable solver system output printing"]:::C

        B2 --> |Set|C1
        B2 --> |Set|C2
        B2 --> |Set|C3
        B2 --> |Set|C4
        B2 --> |Set|C5
        B2 --> |Set|C6
        B2 --> |Set|C7
        B2 --> |Set|C8
        B2 --> |Set|C9
       end

       subgraph SD[ ]
       direction TB
       
        D1["Disable input file listing in output"]:::D
        D2["Disable log file generation"]:::D
        D3["Disable symbol cross-reference listing"]:::D
        D4["Disable symbol list in output"]:::D
           
        C5 --> |Turn off|D1
        C5 --> |Turn off|D2
        C5 --> |Turn off|D3
        C5 --> |Turn off|D4
       end

   end

classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;

classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;

class SA Z; 
class SB Z;
class SC Z;
class SD Z;

%% Default arrow style 
linkStyle default stroke:#FFFFFF,stroke-width:2.5px;

linkStyle 0,1,2 stroke:#004C99,stroke-width:2px;


 ```mermaid
graph LR
    direction LR
    
    subgraph SA["Model Configuration Options"]
    direction LR
    
    A0{{"Model Configuration Options"}}:::A

    A0 --- SB 
    SB --- SC 
    SC --- SD
   
    A0 --> |Define|B1
    A0 --> |Set|B2

        subgraph SB[ ]
        direction TB
               
        B1["Specify input data file name"]:::B
        B2["Global Options"]:::B
       end
        
       subgraph SC[ ]
       direction TB
       
        C1{"Formulation<br>Type"}:::C
        C2{"Status<br>retrieval"}:::C
        C3{"Flexible<br>Demand<br>Equations"}:::C
        C4{"Advanced<br>Reserves"}:::C

        C5{"Frequency<br>Constrains"}:::C


        B2 --> C1
        B2 --> C2
        B2 --> C3
        B2 --> C4
        B2 --> C5
%%        B2 --> C6
%%        B2 --> C7
%%        B2 --> C8
%%        B2 --> C9
       end

       subgraph SD[ ]
       direction TB
       
        D1["LPFormulation = 0"]:::D
        D2["LPFormulation = 1"]:::D
        D3["RetrieveStatus = 0"]:::D
        D4["RetrieveStatus = 1"]:::D
        D5["ActivateFlexibleDemand = 0"]:::D
        D6["ActivateFlexibleDemand = 1"]:::D
        D7["ActivateAdvancedReserves = 0"]:::D
        D8["ActivateAdvancedReserves = 1"]:::D
        D9["FC = 0"]:::D
        D10["FC = 1"]:::D

        C1 --> |MIP|D1
        C1 --> |LP|D2
        C2 --> |Disable|D3
        C2 --> |Enable|D4
        C3 --> |Deactivate|D5
        C3 --> |Activate|D6
        C4 --> |Turn Off|D7
        C4 --> |Turn On|D8
        C5 --> |No|D9
        C5 --> |Yes|D10

       end

    end


classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;

classDef Z1 fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef Z2 fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:12px;

class SA Z1; 
class SB Z1;
class SC Z1;
class SD Z1;

%% Default arrow style 
linkStyle default stroke:#FFFFFF,stroke-width:2.5px;

linkStyle 0,1,2 stroke:#004C99,stroke-width:2px;

 ```mermaid
graph LR
   
    subgraph SA["Set Definitions"]
    direction LR
   
    A0{{"Set Definitions"}}:::A

    A0 --- SB 
    SB --- SC 
    SC --- SD
    SD --- SE
   
    A0 --> B1
    A0 --> B2
    A0 --> B3
    A0 --> B4
    A0 --> B5
    A0 --> B6
    A0 --> B7
    A0 --> B8
    A0 --> B9
                    
        subgraph SB[ ]
        direction TB
               
        B1["Markets"]:::B
        B2["Fuels"]:::B
        B3["Pollutants"]:::B
        B4["Units"]:::B
        B5["Technologies"]:::B
        B6["Simulation Options"]:::B
        B7["Lines"]:::B
        B8["Nodes"]:::B
        B9["Aliases"]:::B
%%        B10["XX"]:::B
%%        B11["XX"]:::B
%%        B12["XX"]:::B
%%        B13["XX"]:::B
%%        B14["XX"]:::B
%%        B15["XX"]:::B
%%        B16["XX"]:::B
%%        B17["XX"]:::B
%%        B18["XX"]:::B

 
       end

       subgraph SC[Power Sector]
       direction TB
       
        C1["Markets"]:::D
        C2["Fuel types"]:::D
        C3["Pollutants"]:::D
        C4["All Units<br>Generation units<br>CHP units<br>Power to X<br>X to Power<br>Boundary sector only units<br>Storage Units (with reservoir)<br>Units with thermal storage<br>Heat only units<br>Hydro technologies<br>Conventional units only<br>Batteries only"]:::D
        C5["Generation technologies<br>Renewable generation technologies<br>Conventional technologies"]:::D
        C6["Hours<br>Subset of simulated hours for one iteration<br>Subset of every simulated hour"]:::D
        C7["Lines<br>Lines between internal zones<br>Lines to the rest of the world"]:::D
        C8["Nodes"]:::D

        B1 --> C1
        B2 --> C2
        B3 --> C3
        B4 --> C4
        B5 --> C5
        B6 --> C6
        B7 --> C7
        B8 --> C8
       end

       subgraph SD[Boundary Sector]
       direction TB::
       
        D1["Boundary sector nodes"]:::D
        D2["Boundary sector lines<br>Boundary sector spillage lines"]:::D
%%        D3["Disable symbol cross-reference listing"]:::D
%%        D4["Disable symbol list in output"]:::D
           
        B8 --> D1
        B7 --> D2
%%        C5 --> |Turn off|D3
%%        C5 --> |Turn off|D4
       end


       subgraph SE[Commons]
       direction TB
       
        E1["Markets<br>Nodes<br>Lines<br>Units<br>Tecnologies<br>Fuels<br>Polluntans<br>Storages<br>Hours<br>Iteration"]:::D
%%        E2["xxx"]:::D
%%        E3["xxx"]:::D
%%        E4["xxx"]:::D
           
        B9 --> E1
       end

   end

classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;





classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef Y fill:#004C99,stroke:#none,stroke-width:6px,color:none,font-weight:bold,font-size:13px;

class SA Z; 
class SB Z;
class SC Y;
class SD Y;
class SE Y;

%% Default arrow style 
linkStyle default stroke:#FFFFFF,stroke-width:2.5px;

linkStyle 0,1,2,3 stroke:#004C99,stroke-width:2px;


 ```mermaid
graph TB

 %% Parameter Definitions
 direction LR

 subgraph S4["Parameter Definitions"]
     
     %% ================= POWER SECTOR =================
     subgraph S4-1["Power Sector"]
         %% --- Units/Operational Parameters ---
         subgraph SC1[ ]
         direction LR
             C1-a["Availability factor<br>CHP power loss factor<br>CHP power-to-heat ratio<br>CHP maximum heat production<br>CHP type<br>Initially committed units<br>Configuration parameter<br>Efficiency<br>Outage factor<br>Minimum part-load<br>Maximum power capacity<br>Initial power output<br>Minimum stable power<br>Maximum ramp down<br>Maximum shutdown ramp<br>Maximum startup ramp<br>Max startup ramp per hour<br>Max shutdown ramp per hour<br>Max ramp up<br>Reserve contribution<br>Technology classification<br>Node location<br>Extended node location<br>Minimum down time<br>Minimum up time<br>Number of units<br>Quick-start units<br>Quick-start power"]:::F
             C1-b{"retrieving<br>status<br>=<br>1?"}:::F --> |Yes|C1-c["Committed calculation"]:::F
         end

         %% --- Economic Parameters ---
         C2-a["Fixed cost<br>Ramp-up cost<br>Ramp-down cost<br>Shutdown cost<br>Startup cost<br>Variable cost<br>Cost of spilled energy<br>Water value cost<br>Storage alert cost<br>Flood control cost<br>Load shedding cost<br>Curtailment cost<br>Transmission price"]:::F

         %% --- Storage Parameters ---
         C3-a["Storage capacity<br>Discharge efficiency<br>Max charging capacity<br>Charging efficiency<br>Self-discharge rate<br>Inflow per hour<br>Outflow per hour<br>Initial storage<br>Storage profile<br>Minimum storage<br>Alert level<br>Flood control<br>Max storage hours<br>Final minimum storage"]:::F 
         
         %% --- Demand/Flexibility Parameters ---
         C4-a["Demand<br>Curtailment limit<br>Load shedding<br>Maximum flexible demand<br>Maximum oversupply<br>Initial accumulated oversupply"]:::F

         %% --- Network Parameters ---
         C6-a["Max line flow<br>Min line flow<br>Max extended line flow<br>Min extended line flow<br>Line-node incidence<br>Extended line-node incidence<br>Power transfer distribution factors (PTDF)"]:::F 

         %% --- Environmental Parameters ---
         C7-a["Max emission<br>Emission rate<br>Fuel type"]:::F

         %% --- Reserve/Frequency Parameters ---
         subgraph SC8[ ]
         direction LR
             C8-b{"MTS<br>=<br>0?"}:::F --> |Yes|C8-c["Inertia constant<br>Inertia limit per hour<br>Droop setting<br>System gain limit<br>FFR gain limit<br>Primary reserve limit<br>FFR limit"]:::F
         end

         %% --- Derived Parameters ---
         C9-a["Maximum load<br>Must-run power"]:::F 

         %% --- Scalars ---
         C10-a["First simulation hour<br>Last simulation hour<br>Last kept hour<br>Day counter<br>Number of days<br>Failure indicator<br>srp parameter<br>nsrp parameter<br>Time step<br>System frequency<br>Max RoCoF<br>Max frequency deviation<br>Max allowed primary reserve"]:::F 
     end

     %% ================= BOUNDARY SECTOR =================
     subgraph S4-2["Boundary Sector"]
         %% --- Storage Parameters ---
         subgraph SC3X[ ]
         direction LR
             C3X-a["Storage capacity<br>Self-discharge rate<br>Storage hours<br>Minimum storage<br>Max charging/discharging power<br>Storage profile<br>Alert level<br>Flood control"]:::F
             C3X-b{"MTS<br>=<br>0?"}:::F --> |Yes|C3X-c["Initial storage<br>Final minimum storage"]:::F
         end

         %% --- Economic Parameters ---
         C2X-a["Storage alert cost<br>Flood control cost<br>Spillage cost<br>Not-served cost"]:::F  

         %% --- Flexibility Parameters ---
         C4X-a["Flexible demand input<br>Initial flexible demand input<br>Max flexible demand capacity<br>Flexible supply input<br>Initial flexible supply input<br>Max flexible supply"]:::F

         %% --- Conversion/Demand Parameters ---
         C5X-a["Power-to-X conversion factor<br>X-sector demand<br>X-to-power conversion factor"]:::F

         %% --- Spillage Parameters ---
         C6X-a["Spillage node mapping<br>Maximum spillage"]:::F 
     end

     %% ================= CONNECTIONS =================
     C02 ---> C2X-a
     C --> C02[Economic<br>&<br>Market Cost<br>Parameters]:::D --> C2-a
     C --> C07[Environmental<br>&<br>Fuel]:::D --> C7-a
     C --> C08[Reserve<br>and<br>Frequency<br>Stability]:::D --> SC8
     C --> C09[Derived<br>Time-Dependent<br>Parameters]:::D --> C9-a
     C --> C010[Scalars<br>&<br>Constants]:::D --> C10-a
     C{{"Parameter Definitions"}}:::C --> C01[Units<br>Operational<br>Parameters]:::D --> SC1
     C --> C04[Flexible Demand<br>&<br>Curtailment]:::D --> C4-a
     C04 ---> C4X-a
     C03 ---> SC3X
     C --> C03[Storage<br>Parameters]:::D --> C3-a
     C --> C06[Network<br>Topology<br>&<br>Limits]:::D --> C6-a
     C06 ---> C6X-a
     C --> C05[Boundary<br>Sector<br>Parameters]:::D ---> C5X-a
 end

 %% Style Definitions
 classDef A fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
 classDef B fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
 classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
 classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:#fff,font-weight:bold,font-size:15px;
 classDef E fill:#004C99,stroke:#none,stroke-width:6px,color:none,font-weight:bold,font-size:15px;
 classDef F fill:#003366,stroke:#003366,stroke-width:6px,color:#fff,font-weight:bold,font-size:10px;
 classDef G fill:#004C99,stroke:#none,stroke-width:6px,color:#fff,font-weight:bold,font-size:10px,stroke-dasharray: 5 5;

  class S4 B;
  class S4-1 E;
  class S4-2 E;
  class SC1 G;
  class SC8 G;
  class SC3X G;

%% Default arrow style 
linkStyle default stroke:#FFFFFF,stroke-width:2.5px;

 ```mermaid
graph LR
 %% Import Data
subgraph S4["Import Data"]
direction LR
%% Root and main branches
A{{"Import Data"}}:::A --> B[/"Load<br>Input data from<br>GDX file"/]:::C
G --- H --- D1 --- D2
    subgraph G[" "]
    direction TB
    B --> C["Sets"]:::C
    B --> D["Parameters"]:::C
    end
C --> C2a
C --> C1a
D --> D1h
D1h --> D1H
D1h --> D2f1
D --> D1a
D1a --> D1a1
D --> D1i
D1i --> D1i1
D --> D1g
D1g --> D1G
D --> D1b
D1b --> D1b1
D1b --> D2a1
D --> D1d
D1d --> D1d1
D1d --> D2c1
D --> D1c
D1c --> D1c1
D1c --> D2b1
D --> D1e
D1e --> D1e1
D1e --> D2D
D --> D1f1
D1f1 --> D2e1
D --> D1f2
D1f2 --> D2e2
    subgraph H[" "]
    direction TB
D1a["Unit<br>Performance"]:::C
D1b["Costs<br>and<br>Economic<br>Factors"]:::C
D1c["Transmission<br>and<br>Network"]:::C
D1d["Geographic<br>Locations"]:::C
D1e["Storage<br>and<br>Resilience"]:::C
D1f1["Energy<br>Conversion"]:::C
D1f2["Flexibility<br>Options"]:::C
D1g["Operational<br>Constraints"]:::C
D1h["Demand<br>and<br>Spillage"]:::C
D1i["Environmental<br>Impact"]:::C
    end
%% ================= POWER SECTOR =================
subgraph D1["Power Sector"]
    direction TB
    %% Power Sector sets
    C1a["Markets<br>Nodes<br>Regions<br>Time periods<br>Technologies<br>Water sources<br>Conversion processes<br>Combined heat and power<br>Custom units<br>Bidding areas<br>Transmission corridors<br>Special zones"]:::C
    %% Unit Performance
D1a1["Technology type<br>Configuration<br>Efficiency<br>Maximum power output<br>Initial power output<br>Minimum operational level<br>Number of units<br>Availability<br>Outage rate<br>Heat and power characteristics"]:::C
    %% Costs and Economic Factors
D1b1["Fixed costs<br>Variable costs<br>Startup costs<br>Shutdown costs<br>Ramp-up costs<br>Ramp-down costs<br>Transmission prices<br>Load shedding costs<br>Curtailment costs<br>Spillage costs<br>Storage alert costs<br>Flood control costs<br>Curtailment costs"]:::C
    %% Transmission & Network
        D1c1["Network connections<br>Maximum flow<br>Minimum flow<br>Power transfer distribution"]:::C
    %% Geographic Locations
        D1d1["Physical location"]:::C
    %% Storage / Resilience
D1e1["Storage capacity<br>Charging capacity<br>Charging efficiency<br>Self-discharge rate<br>Discharging efficiency<br>Minimum storage level<br>Alert level<br>Flood control level<br>Storage duration<br>Storage profile<br>Outflow<br>Initial storage level"]:::C
    %% Operational Constraints
    subgraph D1G[" "]
    direction LR
        D1g1["Maximum ramp-up rate<br>Maximum ramp-down rate<br>Maximum startup ramp rate<br>Maximum shutdown ramp rate<br>Reserve requirements<br>Minimum uptime<br>Minimum downtime"]:::C
        D1g2["System inertia<br>Inertia limits<br>Frequency control gain<br>System gain limit<br>Fast frequency response gain limit<br>Primary reserve limit<br>Fast frequency response limit"]:::C
        D1g2a{"MTS=0?"}:::C --> |Yes|D1g2
    end
    %% Demand and Spillage
    subgraph D1H[" "]
    direction LR
        D1h1["Energy demand<br>Load shedding<br>Energy curtailment"]:::C
        D1h2["Pre-calculated unit commitment status"]:::C
        D1h2a{"Retrieve<br>Status<br>=<br>1?"}:::C --> |Yes|D1h2
    end
    %% Environmental Impact
        D1i1["Fuel type<br>Maximum emissions<br>Emission rate"]:::C
end
%% ================= BOUNDARY SECTOR =================
subgraph D2["Boundary Sector"]
    direction LR
    %% Sector X sets
        C2a["Sector X nodes<br>Sector X regions<br>Special Sector X zones"]:::C
    %% Costs and Economic Factors
        D2a1["Sector X storage alert costs<br>Sector X flood control costs<br>Sector X spillage costs<br>Sector X unserved energy costs"]:::C
    %% Transmission & Network
        D2b1["Sector X spillage nodes<br>Maximum Sector X spillage<br>Sector X network connections<br>Maximum Sector X flow<br>Minimum Sector X flow"]:::C
    %% Geographic Locations
        D2c1["Sector X locations"]:::C
    %% Storage / Resilience
    subgraph D2D[" "]
    direction LR
D2d1["Sector X storage capacity<br>Sector X storage power limit<br>Sector X storage self-discharge<br>Sector X minimum storage level<br>Sector X alert level<br>Sector X flood control level<br>Sector X storage duration<br>Sector X storage profile"]:::C
        D2d2["Initial Sector X storage level"]:::C
        D2d2a{"MTS = 0?"}:::C --> |Yes|D2d2
    end
    %% Energy Conversion
D2e1["Power-to-X conversion rate<br>X-to-power conversion rate"]:::C
    %% Flexibility Options
D2e2["Sector X flexible demand<br>Initial Sector X flexible demand<br>Maximum Sector X flexible demand capacity<br>Sector X flexible supply<br>Initial Sector X flexible supply<br>Maximum Sector X flexible supply"]:::C
    %% Sector X Demand
    D2f1["Sector X demand"]:::C
end
end
%% ================= STYLING =================
classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef Y fill:#004C99,stroke:#none,stroke-width:6px,color:none,font-weight:bold,font-size:15px;
classDef G fill:none,stroke:#none,stroke-width:6px,color:#fff,font-weight:bold,font-size:10px,stroke-dasharray: 5 5;

class S4 Z;
class G Z;
class H Z;
class D1 Y;
class D2 Y;
class D2 Y;
class D1G G;
class D1H G;
class D2D G;

%% Default arrow style
linkStyle default stroke:#FFFFFF,stroke-width:2.5px;
linkStyle 1,2,3 stroke:#004C99,stroke-width:2px;


 ```mermaid
graph LR
 %% Variables Definition------------------------------------------------------------------
subgraph S4["Variables Definition"]
direction LR
%% Root and main branches------------------------------------------------------------------
A{{"Variables<br>Definition"}}:::A
H --- D1 --- D2
A --> D1c
D1c --> D2c1
D1c --> D1c1
A --> D1f
D1f --> D1f3
D1f --> D1F
A --> D1d
D1d --> D2d1
D1d --> D1d1
A --> D1j
D1j --> D1j3
D1j --> D1J
A --> D1a
D1a --> D1A
D1a2 -->  |Yes|D1a3
D1a2 -->  |Np|D1a4
A --> D1i
D1i --> D1i1
A --> D1g
D1g --> D1G
A --> D1b
D1b --> D1b1

A --> D1e
D1e --> D1e1
D1e --> D2E
    subgraph H[" "]
    direction TB
D1c["Reserves<br>and<br>Operational<br>Constraints"]:::C
D1f["Network<br>and<br>Flow"]:::C
D1d["Power<br>Generation<br>and<br>Consumption"]:::C

D1j["Demand<br>and<br>Supply<br>Balancing"]:::C
D1a["Unit<br>Commitment<br>and<br>Operation"]:::C
D1i["Costs<br>and<br>Economic<br>Factors"]:::C
D1g["System<br>Stability<br>and<br>Inertia"]:::C
D1b["System<br>Metrics<br>and<br>Optimization"]:::C
D1e["Storage<br>and<br>Water<br>Management"]:::C
    end
%% ================= POWER SECTOR =================
subgraph D1["Power Sector"]
    direction TB
    %% Reserves and Operational Constraints-----------------------------------------------------------------------------------------
        D1c1["0 <<br>Spinning reserve up<br>Spinning reserve down<br>Non spinning quick start reserve up<br>Total Spinning reserve up<br>maximum power Deficit<br>each plant ramping Deficit<br>ramping down Deficit<br>Power exceeding the demand<br>reserve up Deficit<br>reserve up - non spinning Deficit <br>reserve down Deficit<br>< +Ꝏ"]:::C
    %% System Metrics and Optimization------------------------------------------------------------------
D1b1["-Ꝏ <<br>one optimization period Total system cost <br>Objective Function<br>Optimality Gap<br>Optimizatioz Error<br>Error<br>< +Ꝏ"]:::C
    %% Unit Commitment and Operation--------------------------------------------------------------------------------
        subgraph D1A[" "]
        direction LR
        D1a1["-Ꝏ <<br>Unit committed<br>Unit start up<br>Unit shut down<br>< +Ꝏ"]:::C
        D1a2{"LP<br>=1?"}:::C
        D1a3["0 <<br>Unit committed<br>< 1"]:::C
        D1a4["∈ ℤ<br>Unit committed<br>Unit start up<br>Unit shut down<br>≤<br>Clustered Units Number"]:::C
        end
    %% Power Generation and Consumption------------------------------------------------------------------
        D1d1["0 <<br>Power output<br>P2X units Power consumption<br>Max Power output<br>Min Power output<br>CHP plant Heat output<br>< +Ꝏ"]:::C
    %% System Stability and Inertia----------------------------------------------------------------------------------------
    subgraph D1G[" "]
    direction LR
        D1g1["0 <<br>System Inertia<br>Primary Reserve available<br>System Gain<br>Fast Frequency Reserve available<br>FFR Gain<br>Power Loss<br>< +Ꝏ"]:::C
        D1g2{"MTS<br>=<br>0?"}:::C --> |Yes|D1g1
    end
    %% Costs and Economic Factors
        D1i1["0 ≤<br>starting up Cost<br>shutting down Cost<br>Ramping Up cost<br>Ramping Down cost<br>Hourly system cost<br>< +Ꝏ"]:::C
    %% Network and Flow-------------------------------------------------------------------------------------------------
        subgraph D1F[" "]
        direction LR
        D1f1["0 <<br>Flow through lines<br> < +Ꝏ"]:::C
        D1f2["-Ꝏ <<br>Node injected Power<br>< +Ꝏ"]:::C
        end
    %% Demand and Supply Balancing--------------------------------------------------------------------------------------
        subgraph D1J[" "]
        direction LR
        D1j1["0 <<br>flexible demand Accumulated oversupply<br>Curtailed power<br>2U reserves Curtailed power<br>3U reserves Curtailed power<br>Shed load<br>< +Ꝏ"]:::C
        D1j2["-Ꝏ <<br>flexible demand and baseline Difference<br>Residual Load<br>< +Ꝏ"]:::C
        end
    %% Storage and Water Management------------------------------------------------------------------
D1e1["0 <<br> Storage unit charging input<br>Storage charge level<br>Water reservoir spillage<br>optimization period end Unmet water level <br>timestep end Unmet storage level <br>optimization end Unmet water level <br>timestep end Unmet storage level<br>below alert Unmet water level constraint<br>above flood Unmet water level constraint<br>< +Ꝏ"]:::C
end
%% ================= BOUNDARY SECTOR =================
subgraph D2["Boundary Sector"]
    direction LR
    %% Power Generation and Consumption--------------------------------------------------------------------------------
        D2d1["-Ꝏ <<br>SectorX Power output<br>< +Ꝏ"]:::C
    %% Reserves and Operational Constraints-----------------------------------------------------------------------------------------
        D2c1["0 <<br>flex demand Deficit<br>flex supply Deficit<br>< +Ꝏ"]:::C 
    %% Network and Flow-------------------------------------------------------------------------------------------------
        D1f3["-Ꝏ <<br>Secor X lines Flow<br>< -Ꝏ"]:::C
    %% Demand and Supply Balancing---------------------------------------------------------------------------------------
        D1j3["-Ꝏ <<br>Flexible sectorX demand<br>lexible sectorX supply<br>other sources meet sectorX demand<br>< +Ꝏ"]:::C
    %% Storage and Water Management------------------------------------------------------------------------------------
    subgraph D2E[" "]
    direction LR
    D2e2["0 <<br>sectorX charge Storage level<br>SectorX Storage input/output<br>end optimization Unmet sectorX water level<br>end timestep Unmet sectorX storage<br>water level alert Unmet sectorX<br>water level flood Unmet SectorX<br>SectorX to SectorX'<br>< +Ꝏ"]:::C
    D2e3["0 <<br>Initial sectorX storage Minimum<br>final sectorX storage< +Ꝏ"]:::C
        D2e4{"MTS<br>=<br>1?"}:::C --> |Yes|D2e3
    end
end
end
%% ================= STYLING =================
classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef Y fill:#004C99,stroke:#none,stroke-width:6px,color:none,font-weight:bold,font-size:15px;
classDef G fill:none,stroke:#none,stroke-width:6px,color:#fff,font-weight:bold,font-size:15px,stroke-dasharray: 5 5;

class S4 Z;
class G Z;
class H Z;
class D1 Y;
class D2 Y;
class D2 Y;
class D1A G;
class D1F G;
class D1J G;
class D1G G;
class D1H G;
class D2E G;

%% Default arrow style
linkStyle default stroke:#FFFFFF,stroke-width:4px;
linkStyle 0,1 stroke:#004C99,stroke-width:1px;


 ```mermaid
flowchart LR
subgraph S4[Initial Values Assignment]
direction LR
    A{{Initial Values\nAssignment}}:::A
    A --> |Initial\nCommitment|SB
    A --> |Minimum\nOperating\nLevels|SC 
    A --> |Maximum\nAvailable\nCapacity|SD 
    A --> |Quick-Start\nConfiguration|SE 
    A --> |Must-Run\nRequirements|SG 
    A --> |Reserve\nServices|SH 
    A --> |Flexible\nDemand|SI
    A --> |Simulation\nParameters|SJ


subgraph SA[" "]
direction TB
    
    subgraph SB[" "]
    direction TB
        B[Set Initial Unit Commitment]:::C
        B --> B1[Mark all units offline]:::C
        B1 --> B2[Commit units with initial output]:::C
    end

    subgraph SC[ ]
    direction TB
        C[Define minimum stable power]:::C
    end

    subgraph SD[" "]
    direction TB
        D[Determine max capacity considering availability & outages]:::C
    end

    subgraph SE[" "]
    direction TB
        E[Initialize all units as non-quick-start]:::C
        E --> E2{time to\nmin power\n<= 15?}:::C
        E2 -->|Yes| E3[Enable full quick-start operation]:::C
        E2 -->|No| E4[Keep conventional startup only]:::C
        E3 --> F[Set Operating Ramp Limits]:::C
        E4 --> F
        F --> F1[Define max startup ramps]:::C
        F1 --> F2[Define max shutdown ramps]:::C
    end

    subgraph SG[" "]
    direction TB
        G[Set base must-run power]:::C
        G --> G2{Renewable\nwith\ncurtailment?}:::C
        G2 -->|Yes| G3[Require full capacity must-run]:::C
        G2 -->|No| G4[Keep minimum must-run]:::C
        G3 --> H[Configure Reserve Services]:::C
        G4 --> H
    end

    subgraph SH[" "]
    direction TB
        H1[Set quick-start reserve limits]:::C
    end

    subgraph SI[" "]
    direction TB
        I[Initialize demand flexibility]:::C
        I --> I1[Find max flexible demand per node]:::C
        I1 --> I2[Calculate oversupply limits]:::C
        I2 --> I3[Reset oversupply counters]:::C
    end

    subgraph SJ[" "]
    direction TB
        J[Set simulation time step]:::C
        J --> K[Complete: System Ready for Optimization]:::C
    end
end
end

%% ================= STYLING =================
classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:10px;
classDef Y fill:#004C99,stroke:#none,stroke-width:6px,color:none,font-weight:bold,font-size:10px;
classDef G fill:none,stroke:#none,stroke-width:6px,color:#fff,font-weight:bold,font-size:10px,stroke-dasharray: 5 5;

class S4 Z;
class SA Z;
class SB Y;
class SC A;
class SD A;
class SE Y;
class SG Y;
class SH A;
class SI Y;
class SJ Y;

%% Default arrow style
linkStyle default stroke:#FFFFFF,stroke-width:4px;
%%linkStyle 0,1 stroke:#004C99,stroke-width:1px;

 ```mermaid
graph LR
   
    subgraph SA["Column 1"]
    direction LR
   
    A0{{"Set Definitions"}}:::A

    A0 --- SB 
    SB --- SC 
    SC --- SD
    SD --- SE
    SE --- SF
   
    %% Connections from A0 to B
    A0 --> B1
    A0 --> B2
    A0 --> B3
    A0 --> B4
    A0 --> B5
    A0 --> B6
    A0 --> B7
    A0 --> B8
    A0 --> B9
    A0 --> B10
    A0 --> B11
    A0 --> B12
    A0 --> B13
    A0 --> B14
    A0 --> B15
    A0 --> B16
    A0 --> B17
    A0 --> B18
    A0 --> B19
    A0 --> B20
    A0 --> B21
    A0 --> B22
    A0 --> B23
    A0 --> B24
    A0 --> B25
    A0 --> B26
    A0 --> B27
    A0 --> B28
    A0 --> B29
    A0 --> B30
    A0 --> B31
    A0 --> B32
    A0 --> B33
    A0 --> B34
    A0 --> B35
    A0 --> B36
    A0 --> B37
    A0 --> B38
    A0 --> B39
    A0 --> B40
               
        subgraph SB["Column 2"]
        direction TB

        B1["XXXX"]:::B
        B2["XXXX"]:::B
        B3["XXXX"]:::B
        B4["XXXX"]:::B
        B5["XXXX"]:::B
        B6["XXXX"]:::B
        B7["XXXX"]:::B
        B8["XXXX"]:::B
        B9["XXXX"]:::B
        B10["XXXX"]:::B
        B11["XXXX"]:::B
        B12["XXXX"]:::B
        B13["XXXX"]:::B
        B14["XXXX"]:::B
        B15["XXXX"]:::B
        B16["XXXX"]:::B
        B17["XXXX"]:::B
        B18["XXXX"]:::B
        B19["XXXX"]:::B
        B20["XXXX"]:::B
        B21["XXXX"]:::B
        B22["XXXX"]:::B
        B23["XXXX"]:::B
        B24["XXXX"]:::B
        B25["XXXX"]:::B
        B26["XXXX"]:::B
        B27["XXXX"]:::B
        B28["XXXX"]:::B
        B29["XXXX"]:::B
        B30["XXXX"]:::B
        B31["XXXX"]:::B
        B32["XXXX"]:::B
        B33["XXXX"]:::B
        B34["XXXX"]:::B
        B35["XXXX"]:::B
        B36["XXXX"]:::B
        B37["XXXX"]:::B
        B38["XXXX"]:::B
        B39["XXXX"]:::B
        B40["XXXX"]:::B
       end

       subgraph SC["Column 3"]
       direction TB
       
        C1["XXXX"]:::C
        C2["XXXX"]:::C
        C3["XXXX"]:::C
        C4["XXXX"]:::C
        C5["XXXX"]:::C
        C6["XXXX"]:::C
        C7["XXXX"]:::C
        C8["XXXX"]:::C
        C9["XXXX"]:::C
        C10["XXXX"]:::C
        C11["XXXX"]:::C
        C12["XXXX"]:::C
        C13["XXXX"]:::C
        C14["XXXX"]:::C
        C15["XXXX"]:::C
        C16["XXXX"]:::C
        C17["XXXX"]:::C
        C18["XXXX"]:::C
        C19["XXXX"]:::C
        C20["XXXX"]:::C
        C21["XXXX"]:::C
        C22["XXXX"]:::C
        C23["XXXX"]:::C
        C24["XXXX"]:::C
        C25["XXXX"]:::C
        C26["XXXX"]:::C
        C27["XXXX"]:::C
        C28["XXXX"]:::C
        C29["XXXX"]:::C
        C30["XXXX"]:::C
        C31["XXXX"]:::C
        C32["XXXX"]:::C
        C33["XXXX"]:::C
        C34["XXXX"]:::C
        C35["XXXX"]:::C
        C36["XXXX"]:::C
        C37["XXXX"]:::C
        C38["XXXX"]:::C
        C39["XXXX"]:::C
        C40["XXXX"]:::C

        B1 --> |XXX|C1
        B2 --> |XXX|C2
        B3 --> |XXX|C3
        B4 --> |XXX|C4
        B5 --> |XXX|C5
        B6 --> |XXX|C6
        B7 --> |XXX|C7
        B8 --> |XXX|C8
        B9 --> |XXX|C9
        B10 --> |XXX|C10
        B11 --> |XXX|C11
        B12 --> |XXX|C12
        B13 --> |XXX|C13
        B14 --> |XXX|C14
        B15 --> |XXX|C15
        B16 --> |XXX|C16
        B17 --> |XXX|C17
        B18 --> |XXX|C18
        B19 --> |XXX|C19
        B20 --> |XXX|C20
        B21 --> |XXX|C21
        B22 --> |XXX|C22
        B23 --> |XXX|C23
        B24 --> |XXX|C24
        B25 --> |XXX|C25
        B26 --> |XXX|C26
        B27 --> |XXX|C27
        B28 --> |XXX|C28
        B29 --> |XXX|C29
        B30 --> |XXX|C30
        B31 --> |XXX|C31
        B32 --> |XXX|C32
        B33 --> |XXX|C33
        B34 --> |XXX|C34
        B35 --> |XXX|C35
        B36 --> |XXX|C36
        B37 --> |XXX|C37
        B38 --> |XXX|C38
        B39 --> |XXX|C39
        B40 --> |XXX|C40
       end

       subgraph SD["Column 4"]
       direction TB
       
        D1["XXXX"]:::D
        D2["XXXX"]:::D
        D3["XXXX"]:::D
        D4["XXXX"]:::D
        D5["XXXX"]:::D
        D6["XXXX"]:::D
        D7["XXXX"]:::D
        D8["XXXX"]:::D
        D9["XXXX"]:::D
        D10["XXXX"]:::D
        D11["XXXX"]:::D
        D12["XXXX"]:::D
        D13["XXXX"]:::D
        D14["XXXX"]:::D
        D15["XXXX"]:::D
        D16["XXXX"]:::D
        D17["XXXX"]:::D
        D18["XXXX"]:::D
        D19["XXXX"]:::D
        D20["XXXX"]:::D
        D21["XXXX"]:::D
        D22["XXXX"]:::D
        D23["XXXX"]:::D
        D24["XXXX"]:::D
        D25["XXXX"]:::D
        D26["XXXX"]:::D
        D27["XXXX"]:::D
        D28["XXXX"]:::D
        D29["XXXX"]:::D
        D30["XXXX"]:::D
        D31["XXXX"]:::D
        D32["XXXX"]:::D
        D33["XXXX"]:::D
        D34["XXXX"]:::D
        D35["XXXX"]:::D
        D36["XXXX"]:::D
        D37["XXXX"]:::D
        D38["XXXX"]:::D
        D39["XXXX"]:::D
        D40["XXXX"]:::D

        C1 --> |XXX|D1
        C2 --> |XXX|D2
        C3 --> |XXX|D3
        C4 --> |XXX|D4
        C5 --> |XXX|D5
        C6 --> |XXX|D6
        C7 --> |XXX|D7
        C8 --> |XXX|D8
        C9 --> |XXX|D9
        C10 --> |XXX|D10
        C11 --> |XXX|D11
        C12 --> |XXX|D12
        C13 --> |XXX|D13
        C14 --> |XXX|D14
        C15 --> |XXX|D15
        C16 --> |XXX|D16
        C17 --> |XXX|D17
        C18 --> |XXX|D18
        C19 --> |XXX|D19
        C20 --> |XXX|D20
        C21 --> |XXX|D21
        C22 --> |XXX|D22
        C23 --> |XXX|D23
        C24 --> |XXX|D24
        C25 --> |XXX|D25
        C26 --> |XXX|D26
        C27 --> |XXX|D27
        C28 --> |XXX|D28
        C29 --> |XXX|D29
        C30 --> |XXX|D30
        C31 --> |XXX|D31
        C32 --> |XXX|D32
        C33 --> |XXX|D33
        C34 --> |XXX|D34
        C35 --> |XXX|D35
        C36 --> |XXX|D36
        C37 --> |XXX|D37
        C38 --> |XXX|D38
        C39 --> |XXX|D39
        C40 --> |XXX|D40
       end

       subgraph SE["Column 5"]
       direction TB
       
        E1["XXXX"]:::D
        E2["XXXX"]:::D
        E3["XXXX"]:::D
        E4["XXXX"]:::D
        E5["XXXX"]:::D
        E6["XXXX"]:::D
        E7["XXXX"]:::D
        E8["XXXX"]:::D
        E9["XXXX"]:::D
        E10["XXXX"]:::D
        E11["XXXX"]:::D
        E12["XXXX"]:::D
        E13["XXXX"]:::D
        E14["XXXX"]:::D
        E15["XXXX"]:::D
        E16["XXXX"]:::D
        E17["XXXX"]:::D
        E18["XXXX"]:::D
        E19["XXXX"]:::D
        E20["XXXX"]:::D
        E21["XXXX"]:::D
        E22["XXXX"]:::D
        E23["XXXX"]:::D
        E24["XXXX"]:::D
        E25["XXXX"]:::D
        E26["XXXX"]:::D
        E27["XXXX"]:::D
        E28["XXXX"]:::D
        E29["XXXX"]:::D
        E30["XXXX"]:::D
        E31["XXXX"]:::D
        E32["XXXX"]:::D
        E33["XXXX"]:::D
        E34["XXXX"]:::D
        E35["XXXX"]:::D
        E36["XXXX"]:::D
        E37["XXXX"]:::D
        E38["XXXX"]:::D
        E39["XXXX"]:::D
        E40["XXXX"]:::D

        D1 --> |XXX|E1
        D2 --> |XXX|E2
        D3 --> |XXX|E3
        D4 --> |XXX|E4
        D5 --> |XXX|E5
        D6 --> |XXX|E6
        D7 --> |XXX|E7
        D8 --> |XXX|E8
        D9 --> |XXX|E9
        D10 --> |XXX|E10
        D11 --> |XXX|E11
        D12 --> |XXX|E12
        D13 --> |XXX|E13
        D14 --> |XXX|E14
        D15 --> |XXX|E15
        D16 --> |XXX|E16
        D17 --> |XXX|E17
        D18 --> |XXX|E18
        D19 --> |XXX|E19
        D20 --> |XXX|E20
        D21 --> |XXX|E21
        D22 --> |XXX|E22
        D23 --> |XXX|E23
        D24 --> |XXX|E24
        D25 --> |XXX|E25
        D26 --> |XXX|E26
        D27 --> |XXX|E27
        D28 --> |XXX|E28
        D29 --> |XXX|E29
        D30 --> |XXX|E30
        D31 --> |XXX|E31
        D32 --> |XXX|E32
        D33 --> |XXX|E33
        D34 --> |XXX|E34
        D35 --> |XXX|E35
        D36 --> |XXX|E36
        D37 --> |XXX|E37
        D38 --> |XXX|E38
        D39 --> |XXX|E39
        D40 --> |XXX|E40
       end

       subgraph SF["Column 6"]
       direction TB
       
        F1["XXXX"]:::D
        F2["XXXX"]:::D
        F3["XXXX"]:::D
        F4["XXXX"]:::D
        F5["XXXX"]:::D
        F6["XXXX"]:::D
        F7["XXXX"]:::D
        F8["XXXX"]:::D
        F9["XXXX"]:::D
        F10["XXXX"]:::D
        F11["XXXX"]:::D
        F12["XXXX"]:::D
        F13["XXXX"]:::D
        F14["XXXX"]:::D
        F15["XXXX"]:::D
        F16["XXXX"]:::D
        F17["XXXX"]:::D
        F18["XXXX"]:::D
        F19["XXXX"]:::D
        F20["XXXX"]:::D
        F21["XXXX"]:::D
        F22["XXXX"]:::D
        F23["XXXX"]:::D
        F24["XXXX"]:::D
        F25["XXXX"]:::D
        F26["XXXX"]:::D
        F27["XXXX"]:::D
        F28["XXXX"]:::D
        F29["XXXX"]:::D
        F30["XXXX"]:::D
        F31["XXXX"]:::D
        F32["XXXX"]:::D
        F33["XXXX"]:::D
        F34["XXXX"]:::D
        F35["XXXX"]:::D
        F36["XXXX"]:::D
        F37["XXXX"]:::D
        F38["XXXX"]:::D
        F39["XXXX"]:::D
        F40["XXXX"]:::D

        E1 --> |XXX|F1
        E2 --> |XXX|F2
        E3 --> |XXX|F3
        E4 --> |XXX|F4
        E5 --> |XXX|F5
        E6 --> |XXX|F6
        E7 --> |XXX|F7
        E8 --> |XXX|F8
        E9 --> |XXX|F9
        E10 --> |XXX|F10
        E11 --> |XXX|F11
        E12 --> |XXX|F12
        E13 --> |XXX|F13
        E14 --> |XXX|F14
        E15 --> |XXX|F15
        E16 --> |XXX|F16
        E17 --> |XXX|F17
        E18 --> |XXX|F18
        E19 --> |XXX|F19
        E20 --> |XXX|F20
        E21 --> |XXX|F21
        E22 --> |XXX|F22
        E23 --> |XXX|F23
        E24 --> |XXX|F24
        E25 --> |XXX|F25
        E26 --> |XXX|F26
        E27 --> |XXX|F27
        E28 --> |XXX|F28
        E29 --> |XXX|F29
        E30 --> |XXX|F30
        E31 --> |XXX|F31
        E32 --> |XXX|F32
        E33 --> |XXX|F33
        E34 --> |XXX|F34
        E35 --> |XXX|F35
        E36 --> |XXX|F36
        E37 --> |XXX|F37
        E38 --> |XXX|F38
        E39 --> |XXX|F39
        E40 --> |XXX|F40
       end

   end

%%classDef A fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:25px;
%%classDef B fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:20px;
%%classDef C fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
%%classDef D fill:#003366,stroke:#003366,stroke-width:2px,color:none,font-weight:bold,font-size:10px;

%%classDef Z fill:#004C99,stroke:#004C99,stroke-width:2px,color:none,font-weight:bold,font-size:15px;
%%classDef Y fill:#004C99,stroke:none,stroke-width:2px,color:none,font-weight:bold,font-size:10px;

%%class SA Z; 
%%class SB Z;
%%class SC Y;
%%class SD Y;
%%class SE Y;
%%class SF Y;

%% Default arrow style 
%%linkStyle default stroke:#777

%%linkStyle 0,1,2,3,4 stroke:#004C99,stroke-width:2px;
